In [1]:
pip install ReportLab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 4.4 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import os
import tempfile
from PIL import Image as PILImage
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, KeepTogether, XPreformatted
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# ==========================================
# CARD DATA ARRAYS
# ==========================================

vm_lab_cards = [
    {
        "step": "1. Install VirtualBox",
        "content": (
            "<b>Command:</b>\n"
            "<font name=\"Courier\">sudo apt update\n"
            "sudo apt install virtualbox -y</font>\n\n"
            "<b>Verify:</b>\n"
            "<font name=\"Courier\">VBoxManage --version</font>\n\n"
            "<b>Check virtualization support:</b>\n"
            "<font name=\"Courier\">lscpu | grep -E 'Virtualization|VT-x|AMD-V'</font>"
        )
    },
    {
        "step": "2. Create an isolated Host-Only network",
        "content": (
            "<b>List existing networks:</b>\n"
            "<font name=\"Courier\">VBoxManage list hostonlyifs</font>\n\n"
            "<b>Create one if needed:</b>\n"
            "<font name=\"Courier\">VBoxManage hostonlyif create</font>\n\n"
            "<b>Configure it (Usually vboxnet0):</b>\n"
            "<font name=\"Courier\">VBoxManage hostonlyif ipconfig vboxnet0 --ip 192.168.56.1 --netmask 255.255.255.0</font>\n\n"
            "<b>Recommended Topology:</b>\n"
            "<font name=\"Courier\" size=\"8\">\n"
            "                    YOUR PHYSICAL NETWORK\n"
            "                            |\n"
            "                         Ubuntu Host\n"
            "                            |\n"
            "                    ┌───────┴───────┐\n"
            "                    │   VirtualBox  │\n"
            "                    └───────┬───────┘\n"
            "                            |\n"
            "                    vboxnet0 192.168.56.0/24\n"
            "                            |\n"
            "                  ┌─────────┴─────────┐\n"
            "                  │     pfSense       │\n"
            "                  │ WAN           LAN │\n"
            "                  └─┬──────────────┬──┘\n"
            "                    |              |\n"
            "                  NAT          LAB-LAN (10.10.10.0/24)\n"
            "                    |              |\n"
            "              Internet        ┌─────┴─────┐\n"
            "                              │           │\n"
            "                     Kali (or Laptop)  Targets\n"
            "                        10.10.10.10   10.10.10.x\n"
            "</font>\n\n"
            "<i>Note: Do not put Metasploitable directly onto your physical/home LAN.</i>"
        )
    },
    {
        "step": "3. Create the pfSense VM",
        "content": (
            "Download the current pfSense ISO and create a VM in VirtualBox.\n\n"
            "<b>Specs:</b> RAM: 2 GB | CPU: 2 | Disk: 16 GB\n"
            "<b>Adapter 1 (WAN):</b> Attached to NAT\n"
            "<b>Adapter 2 (LAN):</b> Attached to Host-only Adapter (vboxnet0)\n\n"
            "<b>Configuration:</b>\n"
            "WAN → NAT\n"
            "LAN → vboxnet0 (Set address to <font name=\"Courier\">10.10.10.1/24</font>)\n"
            "Enable DHCP: <font name=\"Courier\">10.10.10.100 - 10.10.10.200</font>"
        )
    },
    {
        "step": "4. Deploy Kali Attacker",
        "content": (
            "<b>Specs:</b> RAM: 4 GB+ | CPU: 2+ | Disk: 40 GB+\n"
            "Network Adapter 1: Attached to Internal Network (Name: LAB-LAN)\n\n"
            "<b>Verify IP and Routing:</b>\n"
            "<font name=\"Courier\">ip addr\n"
            "ip route</font> (Should show default via 10.10.10.1)\n\n"
            "<b>Test pfSense:</b>\n"
            "<font name=\"Courier\">ping -c 4 10.10.10.1</font>"
        )
    },
    {
        "step": "5. Verify Segmentation",
        "content": (
            "From Kali:\n"
            "<font name=\"Courier\">ip route\n"
            "ping -c 4 10.10.10.1\n"
            "sudo nmap -sV 10.10.10.20</font>\n\n"
            "<i>CRITICAL: Do not scan your physical LAN. Check your host's actual network.</i>\n"
            "<i>Your physical interface will likely show 192.168.x.x, while your lab remains 10.10.10.0/24.</i>"
        )
    }
]

pi_honeypot_cards = [
    {
        "step": "1. OS & Host Hardening (Raspberry Pi)",
        "content": (
            "Flash Raspberry Pi OS Lite (64-bit) onto the SD card.\n\n"
            "<b>Move actual SSH to a non-standard port:</b>\n"
            "<font name=\"Courier\">sudo nano /etc/ssh/sshd_config</font>\n\n"
            "Change <font name=\"Courier\">#Port 22</font> to <font name=\"Courier\">Port 2222</font>.\n\n"
            "<b>Restart SSH Service:</b>\n"
            "<font name=\"Courier\">sudo systemctl restart ssh</font>\n\n"
            "<i>Note: This frees up port 22 for the honeypot to trap attackers.</i>"
        )
    },
    {
        "step": "2. Deploy Cowrie SSH/Telnet Honeypot",
        "content": (
            "<b>Run Cowrie container mapping ports 22 and 23:</b>\n"
            "<font name=\"Courier\">docker run -d \\\n"
            "  -p 22:2222/tcp \\\n"
            "  -p 23:2223/tcp \\\n"
            "  -v cowrie-var:/cowrie/cowrie-git/var \\\n"
            "  --name cowrie \\\n"
            "  cowrie/cowrie</font>\n\n"
            "<b>Verify container is running:</b>\n"
            "<font name=\"Courier\">docker ps</font>"
        )
    },
    {
        "step": "3. Log Telemetry & JSON Analysis",
        "content": (
            "Cowrie logs activity in a structured JSON format.\n\n"
            "<b>View live attack logs using jq for clean formatting:</b>\n"
            "<font name=\"Courier\">sudo tail -f /var/lib/docker/volumes/cowrie-var/_data/log/cowrie/cowrie.json | jq</font>\n\n"
            "<b>Look for key event types:</b>\n"
            " - <font name=\"Courier\">cowrie.login.failed</font>\n"
            " - <font name=\"Courier\">cowrie.login.success</font>\n"
            " - <font name=\"Courier\">cowrie.command.input</font>"
        )
    }
]

siem_cards = [
    {
        "step": "1. Provision Central SIEM Server (Wazuh/Splunk)",
        "content": (
            "Deploy an Ubuntu VM with at least 4GB RAM and 50GB storage.\n\n"
            "<b>Install Wazuh Manager (Quickstart):</b>\n"
            "<font name=\"Courier\">curl -sO https://packages.wazuh.com/4.x/wazuh-install.sh\n"
            "sudo bash wazuh-install.sh -a</font>\n\n"
            "<i>Note: Save the auto-generated admin credentials output at the end of the script to access the web dashboard.</i>"
        )
    },
    {
        "step": "2. Deploy Forwarding Agents to Endpoints",
        "content": (
            "Log into the SIEM Web Dashboard and generate an installation command for your target machines.\n\n"
            "<b>Example Agent Deployment (Linux Endpoint):</b>\n"
            "<font name=\"Courier\">WAZUH_MANAGER='10.10.10.50' apt-get install wazuh-agent\n"
            "sudo systemctl daemon-reload\n"
            "sudo systemctl enable wazuh-agent\n"
            "sudo systemctl start wazuh-agent</font>"
        )
    },
    {
        "step": "3. Configure Log Ingestion",
        "content": (
            "Edit the agent configuration file on the endpoint to forward specific logs.\n\n"
            "<b>Configuration Path:</b> <font name=\"Courier\">/var/ossec/etc/ossec.conf</font>\n\n"
            "Ensure system logs like auth.log (Linux) or Event Viewer (Windows) are included in the <font name=\"Courier\">&lt;localfile&gt;</font> block to track authentication events."
        )
    },
    {
        "step": "4. Write Custom Correlation Rules",
        "content": (
            "On the SIEM server, create an alert rule for brute-force attacks.\n\n"
            "<b>Example Logic (5 failed logins in 1 minute):</b>\n"
            "<font name=\"Courier\">&lt;rule id=\"100001\" level=\"10\" frequency=\"5\" timeframe=\"60\"&gt;\n"
            "  &lt;if_matched_sid&gt;5716&lt;/if_matched_sid&gt;\n"
            "  &lt;description&gt;Multiple failed SSH logins detected&lt;/description&gt;\n"
            "&lt;/rule&gt;</font>"
        )
    },
    {
        "step": "5. Generate Simulated Attacks to Validate",
        "content": (
            "From your attacker VM, launch a brute-force attack against the endpoint.\n\n"
            "<b>Simulate Attack with Hydra:</b>\n"
            "<font name=\"Courier\">hydra -l root -P passwords.txt ssh://10.10.10.20</font>\n\n"
            "Check your SIEM dashboard. You should see a Level 10 Alert trigger based on the correlation rule from Step 4."
        )
    }
]

python_log_cards = [
    {
        "step": "1. Read System Logs via Python",
        "content": (
            "Write a script to open and read local system logs.\n\n"
            "<b>Python Code:</b>\n"
            "<font name=\"Courier\">def read_logs(filepath):\n"
            "    with open(filepath, 'r') as file:\n"
            "        return file.readlines()\n\n"
            "logs = read_logs('/var/log/auth.log')</font>"
        )
    },
    {
        "step": "2. Extract IPs using Regex",
        "content": (
            "Utilize the 're' module to extract IP addresses associated with failed logins.\n\n"
            "<b>Python Code:</b>\n"
            "<font name=\"Courier\">import re\n"
            "ip_pattern = r\"Failed password.*from ([0-9\\.]+)\"\n"
            "suspicious_ips = []\n"
            "for line in logs:\n"
            "    match = re.search(ip_pattern, line)\n"
            "    if match:\n"
            "        suspicious_ips.append(match.group(1))</font>"
        )
    },
    {
        "step": "3. Count Frequencies with Dictionaries",
        "content": (
            "Check extracted IPs to count attack frequencies.\n\n"
            "<b>Python Code:</b>\n"
            "<font name=\"Courier\">from collections import Counter\n"
            "ip_counts = Counter(suspicious_ips)\n"
            "for ip, count in ip_counts.items():\n"
            "    if count &gt; 5:\n"
            "        print(f\"ALERT: {ip} failed {count} times\")</font>"
        )
    },
    {
        "step": "4. File Integrity Monitoring (FIM)",
        "content": (
            "Hash critical system files and compare against known-good hashes to detect changes.\n\n"
            "<b>Python Code:</b>\n"
            "<font name=\"Courier\">import hashlib\n"
            "def hash_file(filepath):\n"
            "    hasher = hashlib.sha256()\n"
            "    with open(filepath, 'rb') as f:\n"
            "        hasher.update(f.read())\n"
            "    return hasher.hexdigest()\n\n"
            "current_hash = hash_file('/etc/passwd')</font>"
        )
    },
    {
        "step": "5. Output Formatted CSV Report",
        "content": (
            "Export the collected anomalies to a CSV file.\n\n"
            "<b>Python Code:</b>\n"
            "<font name=\"Courier\">import csv\n"
            "with open('anomaly_report.csv', 'w', newline='') as file:\n"
            "    writer = csv.writer(file)\n"
            "    writer.writerow(['IP Address', 'Failed Attempts'])\n"
            "    for ip, count in ip_counts.items():\n"
            "        writer.writerow([ip, count])</font>"
        )
    }
]

phishing_cards = [
    {
        "step": "1. Deploy Gophish Framework",
        "content": (
            "Set up an open-source phishing framework.\n\n"
            "<b>Commands:</b>\n"
            "<font name=\"Courier\">unzip gophish-vX.X-linux-64bit.zip\n"
            "cd gophish\n"
            "sudo chmod +x gophish\n"
            "sudo ./gophish</font>\n\n"
            "Access the admin interface at <font name=\"Courier\">https://127.0.0.1:3333</font>."
        )
    },
    {
        "step": "2. Configure Sending Profile & Target Groups",
        "content": (
            "<b>SMTP Setup:</b> Link Gophish to an SMTP relay or a local testing mail server (like Mailhog) to send emails safely without being flagged by real providers.\n\n"
            "<b>Target Groups:</b> Import a CSV containing the emails and names of your consenting test group (family members or students)."
        )
    },
    {
        "step": "3. Design Email Template & Landing Page",
        "content": (
            "<b>Email Template:</b> Design a benign email mimicking a common service (e.g., an 'Urgent Password Reset' for an internal network portal).\n\n"
            "<b>Landing Page:</b> Clone a login page using the Gophish site cloner to serve as the harmless payload destination. Ensure 'Capture Submitted Data' is enabled (but configure it to only capture passwords for metrics, or discard them instantly for privacy)."
        )
    },
    {
        "step": "4. Launch Campaign & Track Metrics",
        "content": (
            "Launch the campaign and monitor the live dashboard.\n\n"
            "<b>Track the following events:</b>\n"
            "1. Email Sent\n"
            "2. Email Opened (tracked via invisible tracking pixel)\n"
            "3. Link Clicked\n"
            "4. Submitted Data (Credential entry rates)"
        )
    },
    {
        "step": "5. Present Educational Remediation",
        "content": (
            "Provide a training module explaining the specific red flags the users missed.\n\n"
            "<b>Key Talking Points:</b>\n"
            "- Inspecting the true sender email domain (Domain spoofing).\n"
            "- Hovering over links to verify destinations before clicking.\n"
            "- Identifying psychological triggers like false urgency or fear.\n"
            "- Avoiding unexpected or unverified attachments."
        )
    }
]

network_monitor_cards = [
    {
        "step": "1. Sweep the Local Subnet",
        "content": (
            "Use arp-scan to discover devices on the local Layer 2 network.\n\n"
            "<b>Command:</b>\n"
            "<font name=\"Courier\">sudo apt install arp-scan -y\n"
            "sudo arp-scan --localnet &gt; current_scan.txt</font>"
        )
    },
    {
        "step": "2. Output Discovered Devices",
        "content": (
            "Parse the arp-scan output to extract just the MAC and IP addresses, saving them to a baseline state file.\n\n"
            "<b>Command:</b>\n"
            "<font name=\"Courier\">awk '/[0-9A-Fa-f]{2}:/ {print $1, $2}' current_scan.txt &gt; discovered_devices.txt</font>"
        )
    },
    {
        "step": "3. Create a 'Known Devices' Whitelist",
        "content": (
            "Manually review <font name=\"Courier\">discovered_devices.txt</font> and copy authorized MAC addresses into a new file named <font name=\"Courier\">whitelist.txt</font>.\n\n"
            "<b>Format (whitelist.txt):</b>\n"
            "<font name=\"Courier\">aa:bb:cc:dd:ee:ff\n"
            "11:22:33:44:55:66</font>"
        )
    },
    {
        "step": "4. Automate with Cron Job",
        "content": (
            "Write a shell script (<font name=\"Courier\">monitor.sh</font>) to run the scan, extract MACs, and check against the whitelist. Set it to run every 5 minutes.\n\n"
            "<b>Crontab Entry:</b>\n"
            "<font name=\"Courier\">crontab -e\n"
            "*/5 * * * * /path/to/monitor.sh</font>"
        )
    },
    {
        "step": "5. Trigger Alert on Unknown MAC",
        "content": (
            "In your <font name=\"Courier\">monitor.sh</font> script, diff the live scan against the whitelist and send a webhook if an anomaly is found.\n\n"
            "<b>Alert Logic:</b>\n"
            "<font name=\"Courier\">if ! grep -q \"$MAC\" whitelist.txt; then\n"
            "    curl -X POST -H \"Content-Type: application/json\" \\\n"
            "    -d '{\"content\":\"Unknown device detected: '\"$MAC\"'\"}' \\\n"
            "    https://your-webhook-url.com/api\n"
            "fi</font>"
        )
    }
]

linux_hardening_cards = [
    {
        "step": "1. Restrict SSH Access",
        "content": (
            "Disable root SSH login and enforce public-key authentication.\n\n"
            "<b>Configure sshd_config:</b>\n"
            "<font name=\"Courier\">sudo nano /etc/ssh/sshd_config</font>\n\n"
            "<b>Update Directives:</b>\n"
            "<font name=\"Courier\">PermitRootLogin no\n"
            "PasswordAuthentication no</font>\n\n"
            "<b>Apply Changes:</b>\n"
            "<font name=\"Courier\">sudo systemctl restart ssh</font>"
        )
    },
    {
        "step": "2. Configure UFW (Uncomplicated Firewall)",
        "content": (
            "Block all ports except those explicitly required (e.g., SSH and Web traffic).\n\n"
            "<b>Commands:</b>\n"
            "<font name=\"Courier\">sudo apt install ufw -y\n"
            "sudo ufw default deny incoming\n"
            "sudo ufw default allow outgoing\n"
            "sudo ufw allow OpenSSH\n"
            "sudo ufw allow 80/tcp\n"
            "sudo ufw allow 443/tcp\n"
            "sudo ufw enable</font>"
        )
    },
    {
        "step": "3. Disable Unused System Services",
        "content": (
            "Reduce the attack surface by identifying and disabling unnecessary systemd background processes.\n\n"
            "<b>List running services:</b>\n"
            "<font name=\"Courier\">systemctl list-unit-files --state=enabled</font>\n\n"
            "<b>Disable target service:</b>\n"
            "<font name=\"Courier\">sudo systemctl stop &lt;service_name&gt;\n"
            "sudo systemctl disable &lt;service_name&gt;</font>"
        )
    },
    {
        "step": "4. Deploy Fail2Ban for Brute Force Protection",
        "content": (
            "Install Fail2Ban to automatically ban IP addresses that repeatedly fail authentication.\n\n"
            "<b>Commands:</b>\n"
            "<font name=\"Courier\">sudo apt install fail2ban -y\n"
            "sudo cp /etc/fail2ban/jail.conf /etc/fail2ban/jail.local\n"
            "sudo systemctl enable fail2ban\n"
            "sudo systemctl start fail2ban</font>"
        )
    },
    {
        "step": "5. Configure Unattended Upgrades",
        "content": (
            "Ensure the server automatically installs security patches.\n\n"
            "<b>Commands:</b>\n"
            "<font name=\"Courier\">sudo apt install unattended-upgrades -y\n"
            "sudo dpkg-reconfigure -plow unattended-upgrades</font>\n\n"
            "Select <b>Yes</b> to automatically download and install stable updates."
        )
    }
]

vuln_reporting_cards = [
    {
        "step": "1. Deploy Deliberately Vulnerable Web App (DVWA)",
        "content": (
            "Spin up an isolated container hosting a vulnerable web application for safe testing.\n\n"
            "<b>Command:</b>\n"
            "<font name=\"Courier\">docker run --rm -it -p 80:80 vulnerables/web-dvwa</font>\n\n"
            "Access the application at <font name=\"Courier\">http://localhost</font> and log in with default credentials (<font name=\"Courier\">admin/password</font>)."
        )
    },
    {
        "step": "2. Execute and Document an Exploit",
        "content": (
            "Navigate to the Cross-Site Scripting (XSS) module and execute a safe payload.\n\n"
            "<b>Payload:</b>\n"
            "<font name=\"Courier\">&lt;script&gt;alert('Vulnerability Executed')&lt;/script&gt;</font>\n\n"
            "Take screenshots of the input field and the resulting alert box popup to use as evidence for your Proof of Concept (PoC)."
        )
    },
    {
        "step": "3. Calculate Preliminary CVSS Score",
        "content": (
            "Use the Common Vulnerability Scoring System (CVSS) calculator to quantify the risk.\n\n"
            "<b>Evaluate Base Metrics:</b>\n"
            "- Attack Vector: Network\n"
            "- Attack Complexity: Low\n"
            "- Privileges Required: None (if unauthenticated)\n"
            "- User Interaction: Required (for Reflected XSS)\n"
            "- Impact (C/I/A): Low/Low/None"
        )
    },
    {
        "step": "4. Draft Formal Disclosure Report",
        "content": (
            "Structure the report so developers and management can clearly understand the flaw.\n\n"
            "<b>Required Sections:</b>\n"
            "1. <b>Executive Summary:</b> High-level business impact.\n"
            "2. <b>Vulnerability Details:</b> Affected URL and parameter.\n"
            "3. <b>Proof of Concept:</b> Step-by-step reproduction instructions with screenshots.\n"
            "4. <b>Impact Analysis:</b> How an attacker could leverage this (e.g., session hijacking)."
        )
    },
    {
        "step": "5. Provide Remediation Strategies",
        "content": (
            "Recommend specific, actionable fixes for the engineering team.\n\n"
            "<b>Example Remediation (XSS):</b>\n"
            "Ensure all user-supplied input is sanitized and properly encoded before being rendered in the DOM. Utilize functions like <font name=\"Courier\">htmlspecialchars()</font> in PHP or implement a strict Content Security Policy (CSP) header."
        )
    }
]

# ==========================================
# MAIN DICTIONARY
# ==========================================

cyber_projects = {
    "VM Home Cybersecurity Lab": {
        "Method": "Virtualization and Network Isolation",
        "Diagram_Prompt": "An isometric 3D architectural diagram of a computer network. A glowing hypervisor base supports several floating, isolated glass cubes representing virtual machines. A central glowing red firewall connects the cubes. Clean white background, technical blueprint style, vector art.",
        "Cards": vm_lab_cards
    },
    "Raspberry Pi Honeypot": {
        "Method": "Decoy Systems and Telemetry Collection",
        "Diagram_Prompt": "A cyberpunk-style neon diagram. A glowing Raspberry Pi sits in the center acting as a trap, absorbing malicious red network packets raining down from a dark internet cloud. Cyberpunk aesthetic, bright cyan and magenta accents.",
        "Cards": pi_honeypot_cards
    },
    "SIEM Tool Setup (Wazuh/Splunk)": {
        "Method": "Centralized Log Aggregation and Rule-Based Alerting",
        "Diagram_Prompt": "A minimalist, flat-design data flow diagram. Multiple laptop and server icons on the periphery send streams of glowing binary data toward a massive central glowing brain/server icon in the middle. The central server emits a bright yellow warning symbol. High contrast, modern tech aesthetic.",
        "Cards": siem_cards
    },
    "Python Log Analysis Script": {
        "Method": "Regex Parsing and File I/O Automation",
        "Diagram_Prompt": "A flowchart-style algorithmic diagram. A mechanical Python snake icon is scanning a scrolling text document. Magnifying glasses highlight extracted IP addresses. Arrows point from the extracted data into a final red warning report. Clean tech-manual illustration style, black and yellow color palette.",
        "Cards": python_log_cards
    },
    "Phishing-Awareness Project": {
        "Method": "Simulated Social Engineering and Educational Metrics",
        "Diagram_Prompt": "A user-centric behavioral flowchart. An envelope icon travels from a shadowed hacker figure to a confused office worker. The path splits into two branches: a red line leading to a hooked fish icon, and a green line leading to a graduation cap icon. Infographic style, corporate flat vector art.",
        "Cards": phishing_cards
    },
    "Network-Monitoring Project": {
        "Method": "ARP Scanning and Baseline State Comparison",
        "Diagram_Prompt": "A tactical radar screen diagram. A sweeping circular radar UI in bright green displays various friendly dots. One dot is flashing bright red, identified as an 'unknown device'. Surrounding the radar are floating terminal windows displaying MAC addresses. UI/UX interface design style.",
        "Cards": network_monitor_cards
    },
    "Linux Server Hardening": {
        "Method": "Attack Surface Reduction and Access Control",
        "Diagram_Prompt": "A fortress metaphor diagram. A Linux penguin logo stands in the center, surrounded by concentric layers of defense: a brick wall (firewall), a glowing padlock (SSH keys), and a metal shield (Fail2Ban). Isometric vector art, vibrant colors, clean edges.",
        "Cards": linux_hardening_cards
    },
    "Responsible Vulnerability Reporting": {
        "Method": "Standardized Security Disclosure and Documentation",
        "Diagram_Prompt": "A document lifecycle diagram. It starts with a glowing green magnifying glass inspecting lines of code, an arrow points to a highly structured clinical clipboard with warning stamps, and a final arrow points to a patched, glowing gold code block. Corporate cybersecurity whitepaper style, monochromatic blue with red accents.",
        "Cards": vuln_reporting_cards
    }
}

# ==========================================
# GRADIO & REPORTLAB LOGIC
# ==========================================

def generate_markdown_cards(project_name):
    """Parses the formatting strings into clean Markdown for the Gradio UI."""
    if project_name not in cyber_projects or "Cards" not in cyber_projects[project_name]:
        return "Card format not configured for this project."

    md_output = ""
    for card in cyber_projects[project_name]["Cards"]:
        md_output += f"### {card['step']}\n"
        content = card['content']
        content = content.replace('<b>', '**').replace('</b>', '**')
        content = content.replace('<i>', '*').replace('</i>', '*')
        content = content.replace('<font name="Courier" size="8">', '```text\n')
        content = content.replace('<font name="Courier">', '```bash\n')
        content = content.replace('</font>', '\n```')

        # UI fix specifically to render the Python script indents nicely
        content = content.replace('    ', '&nbsp;&nbsp;&nbsp;&nbsp;')
        md_output += f"{content}\n\n---\n"
    return md_output

def update_ui(project_name):
    details = cyber_projects.get(project_name, cyber_projects["VM Home Cybersecurity Lab"])
    return details["Method"], details["Diagram_Prompt"], generate_markdown_cards(project_name), None

def compile_pdf(pil_image, project_name):
    if pil_image is None or project_name not in cyber_projects:
        return None

    try:
        temp_dir = tempfile.gettempdir()
        temp_img_path = os.path.join(temp_dir, "uploaded_diagram.png")

        if pil_image.mode in ("RGBA", "P"):
            pil_image = pil_image.convert("RGB")
        pil_image.save(temp_img_path, format="PNG")

        pdf_filename = f"{project_name.replace(' ', '_')}_Card_Report.pdf"
        pdf_path = os.path.join(temp_dir, pdf_filename)

        doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=30, leftMargin=30, topMargin=30, bottomMargin=30)
        styles = getSampleStyleSheet()

        card_title_style = ParagraphStyle(
            name='CardTitle',
            parent=styles['Heading3'],
            textColor=colors.whitesmoke,
            backColor=colors.HexColor('#2c3e50'),
            spaceBefore=0,
            spaceAfter=10,
            leftIndent=5,
            borderPadding=5
        )

        card_body_style = ParagraphStyle(
            name='CardBody',
            parent=styles['BodyText'],
            fontSize=10,
            leading=14,
            leftIndent=10,
            rightIndent=10,
            spaceBefore=5,
            spaceAfter=5
        )

        story = []
        story.append(Paragraph(f"Cybersecurity Lab Architecture: {project_name}", styles['Title']))
        story.append(Spacer(1, 15))

        img_w, img_h = pil_image.size
        max_w, max_h = 500.0, 280.0
        scale = min(max_w / img_w, max_h / img_h)
        rl_img = RLImage(temp_img_path, width=img_w * scale, height=img_h * scale)
        rl_img.hAlign = 'CENTER'
        story.append(rl_img)
        story.append(Spacer(1, 20))

        for card in cyber_projects[project_name]["Cards"]:
            title_p = Paragraph(card["step"], card_title_style)
            body_p = XPreformatted(card["content"], card_body_style)

            card_data = [[title_p], [body_p]]

            card_table = Table(card_data, colWidths=[520])
            card_table.setStyle(TableStyle([
                ('BOX', (0,0), (-1,-1), 1.5, colors.HexColor('#2c3e50')),
                ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2c3e50')),
                ('BACKGROUND', (0,1), (-1,1), colors.HexColor('#f8f9fa')),
                ('VALIGN', (0,0), (-1,-1), 'TOP'),
                ('BOTTOMPADDING', (0,1), (-1,1), 10),
                ('TOPPADDING', (0,1), (-1,1), 10),
            ]))

            story.append(KeepTogether(card_table))
            story.append(Spacer(1, 15))

        doc.build(story)
        return pdf_path

    except Exception as e:
        print(f"Error compiling PDF: {e}")
        return None

with gr.Blocks(theme=gr.themes.Monochrome()) as app:
    gr.Markdown("# Cybersecurity Project Planner & Diagram Generator")

    with gr.Row():
        with gr.Column(scale=1):
            project_dropdown = gr.Dropdown(
                choices=list(cyber_projects.keys()),
                label="Select Project Choice",
                value="VM Home Cybersecurity Lab"
            )
            method_box = gr.Textbox(label="Methodology", lines=1, interactive=False)
            prompt_box = gr.Textbox(label="AI Generation Prompt (Copy to Midjourney/DALL-E)", lines=4, interactive=False)

            gr.Markdown("### Step 1: Paste or Upload Diagram")
            diagram_image = gr.Image(label="Paste Generated Diagram Here", interactive=True, type="pil")

            gr.Markdown("### Step 2: Download Compiled PDF")
            pdf_output = gr.File(label="Download PDF Report", interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("### Step-by-Step Implementation Cards")
            card_display = gr.Markdown()

    project_dropdown.change(
        fn=update_ui,
        inputs=project_dropdown,
        outputs=[method_box, prompt_box, card_display, pdf_output]
    )

    diagram_image.change(
        fn=compile_pdf,
        inputs=[diagram_image, project_dropdown],
        outputs=[pdf_output]
    )

    app.load(fn=update_ui, inputs=project_dropdown, outputs=[method_box, prompt_box, card_display, pdf_output])

if __name__ == "__main__":
    app.launch(share=True, debug=True)

/tmp/ipykernel_1490/4156006623.py:629: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://79cabdfafd9ca38886.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
